# pxr-reduce tutorial

A guided walkthrough of loading, examining, processing, reducing, and exporting
polarized X-ray reflectivity data.

This notebook is **self-contained**: the next section generates a small synthetic
dataset so everything runs without real data. To use your own data, skip the
"Generate synthetic data" section and set `data_folder` to your `.fits` directory.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pxr_reduce import PXRLoader, ReductionConfig, ReducedDataset
from pxr_reduce.viewer import frame_figure, select_indices

## 2. Generate synthetic data (skip if you have real data)

Writes a handful of synthetic `.fits` frames matching the BL 11.0.1.2 layout
(metadata in HDU 0, image in HDU 2): two direct-beam (i0) frames followed by a
descending-intensity reflectivity scan.

In [ ]:
from astropy.io import fits


def beam_image(peak, size=81, center=(40, 40)):
    yy, xx = np.mgrid[0:size, 0:size]
    r2 = (yy - center[0]) ** 2 + (xx - center[1]) ** 2
    return 5.0 + peak * np.exp(-r2 / (2 * 4.0**2))


def frame_header(sam_th, sam_z, energy=250.0, polarization=100.0):
    return {
        "Beamline Energy": energy, "EPU Polarization": polarization,
        "Sample Theta": sam_th, "CCD Theta": 2 * sam_th,
        "Sample X": 0.0, "Sample Y": 0.0, "Sample Z": sam_z,
        "EXPOSURE": 1.0, "Higher Order Suppressor": 5.0,
        "Upstream JJ Vert Aperture": 0.1, "Upstream JJ Horz Aperture": 0.1,
        "Beam Current": 500.0, "AI 3 Izero": 1.0,
    }


def write_fits(path, image, header):
    primary = fits.PrimaryHDU()
    for k, v in header.items():
        primary.header[k] = v
    fits.HDUList([
        primary,
        fits.ImageHDU(data=np.zeros((2, 2))),
        fits.ImageHDU(data=image.astype(np.int32)),
    ]).writeto(path, overwrite=True)


data_folder = Path("_tutorial_data")
data_folder.mkdir(exist_ok=True)

peaks = [12000, 12000, 6000, 3000, 1500, 800, 400, 200]
sam_z = [0, 0, 1, 1, 1, 1, 1, 1]
sam_th = [0, 0, 1, 2, 3, 4, 5, 6]
for i, (pk, z, th) in enumerate(zip(peaks, sam_z, sam_th)):
    write_fits(data_folder / f"Tutorial_{i}.fits", beam_image(pk), frame_header(th, z))

print(f"Wrote {len(peaks)} frames to {data_folder.resolve()}")

## 3. Point at the data

For **your own data**, set `data_folder` to your directory here.

In [ ]:
# data_folder = Path("D:/ALS/2020 Nov/MF114A/spol/250eV")  # <- your data
files = sorted(data_folder.glob("*.fits"))
print(f"Found {len(files)} files")
files[:3]

## 4. Configure the reduction

Every reduction parameter lives in `ReductionConfig`. Defaults are sensible; here
we set a small ROI to match the synthetic beam. See the Configuration reference
for every option.

In [ ]:
config = ReductionConfig(
    detector="cmos_11012",
    roi_height=20,
    roi_width=20,
    trim_x=5,
    trim_y=5,
    mask_threshold=100,
)
config

## 5. Load

Creating the loader reads the FITS *headers* and builds the scalar metadata table.
Images are **not** loaded yet — they are read lazily on demand.

In [ ]:
loader = PXRLoader(files, config)
print(loader)
loader.data.head()

## 6. Examine metadata

Query frames by exact value or an inclusive `(low, high)` range.

In [ ]:
loader.query(sam_th=(1.0, 4.0))[["fits_index", "energy", "polarization", "sam_th", "q"]]

## 7. Process

`process()` builds the drift-tolerant beam mask and integrates every frame into
background-subtracted counts with propagated uncertainty. New columns
(`counts_spot`, `counts_dark`, `counts_refl`, `counts_err`, `beam_spot`, ...) are
added to `loader.data`.

In [ ]:
loader.process()
loader.data[["fits_index", "sam_th", "counts_spot", "counts_dark", "counts_refl", "is_saturated"]].head()

## 8. Inspect a frame

`frame_figure` renders a frame with the mask, beam position, and beam/dark ROI
boxes, plus a panel of scalar readouts. Great for confirming the ROI lands on the
beam.

In [ ]:
idx = select_indices(loader, sam_th=(1.0, 4.0))[0]
fig = frame_figure(loader, idx)
fig

For live cycling through frames (outside this notebook, in a windowed session):

```python
from pxr_reduce.viewer import FrameBrowser
FrameBrowser(loader, sam_th=(0.0, 6.0)).show()
```

## 9. Reduce

`reduce()` normalizes to the direct beam (i0), detects stitch boundaries, fits
scale factors, and returns the 1D reflectivity curve.

In [ ]:
refl = loader.reduce()
refl

A **quick** reduction skips stitch scaling — useful for a fast look that avoids
overlap-scaling pitfalls:

In [ ]:
quick = loader.reduce(apply_scale=False)
quick.head()

## 10. Plot R vs q

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(refl["q"], refl["R"], yerr=refl["R_err"], fmt="o-", ms=4, capsize=2)
ax.set_yscale("log")
ax.set_xlabel(r"q [$\AA^{-1}$]")
ax.set_ylabel("Reflectivity")
ax.set_title("Reduced reflectivity")
plt.show()

## 11. Export

Wrap the loader in a `ReducedDataset` and save. This writes a tab-delimited
`.dat` with a full provenance header, plus one I-vs-q PNG per (energy,
polarization) in a sibling `*_plots/` folder.

In [ ]:
dataset = ReducedDataset.from_loader(loader)
result = dataset.save("results/Tutorial.dat")
print("Wrote:", result["dat"])
for p in result["plots"]:
    print("Plot :", p)

# Peek at the header
print("\n".join(result["dat"].read_text().splitlines()[:20]))

## 12. Combine datasets (e.g. two polarizations)

Reduce each polarization into its own `ReducedDataset`, then combine — the merged
header preserves the provenance of both sources.

```python
spol = ReducedDataset.from_loader(loader_spol)
ppol = ReducedDataset.from_loader(loader_ppol)
combined = spol.combine(ppol)
combined.save("results/MF114A_both.dat")
```

## Next steps

- Try `ReductionConfig(roi_from_beam_fit=True)` to size the ROI from the direct
  beam automatically.
- See the **How-to guide** for the command-line workflow (`pxr-reduce run`).
- See the **Configuration reference** for every parameter.
- See the **API reference** for all functions and classes.